In [28]:
# -- coding: utf-8 --
"""
Script d'agrégation mensuelle de données météorologiques horaires
Pour Google Colab - Support CSV et Excel
"""

import pandas as pd
import numpy as np
from google.colab import files
import io
import os
import xml.etree.ElementTree as ET
from xlrd.biffh import XLRDError # Import XLRDError

def _parse_xml_spreadsheet_to_dataframe(xml_content_str):
    """
    Parse an XML Spreadsheet 2003 content string into a pandas DataFrame.
    Assumes the first non-empty row contains headers.
    """
    try:
        # Handle BOM if present
        if xml_content_str.startswith('\ufeff'):
            xml_content_str = xml_content_str[1:]

        root = ET.fromstring(xml_content_str)
    except ET.ParseError as e:
        raise ValueError(f"Erreur de parsing XML: {e}")

    # Namespaces
    ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}

    dataframes = []
    for worksheet in root.findall('.//ss:Worksheet', ns):
        rows = worksheet.findall('.//ss:Row', ns)
        if not rows:
            continue # Skip empty worksheets

        header = []
        data_rows = []
        max_cols_seen = 0

        # Process header row (first row)
        header_row = rows[0]
        current_col_idx_header = 0
        for cell in header_row.findall('./ss:Cell', ns):
            index_attr = cell.get(f'{{{ns["ss"]}}}Index')
            if index_attr:
                current_col_idx_header = int(index_attr) - 1
            data_element = cell.find('./ss:Data', ns)
            header_text = data_element.text.strip() if data_element is not None and data_element.text is not None else f'Column_{current_col_idx_header}'
            header.append(header_text)
            current_col_idx_header += 1
        # Ensure header has unique names if duplicates exist (e.g., empty headers)
        seen_headers = {}
        processed_header = []
        for h in header:
            if h in seen_headers:
                seen_headers[h] += 1
                processed_header.append(f"{h}_{seen_headers[h]}")
            else:
                seen_headers[h] = 1
                processed_header.append(h)
        header = processed_header
        max_cols_seen = len(header)


        # Process data rows (from second row onwards)
        for row_elem in rows[1:]:
            row_data_dict = {}
            current_col_idx = 0
            for cell_elem in row_elem.findall('./ss:Cell', ns):
                index_attr = cell_elem.get(f'{{{ns["ss"]}}}Index')
                if index_attr:
                    current_col_idx = int(index_attr) - 1

                data_element = cell_elem.find('./ss:Data', ns)
                cell_value = data_element.text if data_element is not None else ''
                row_data_dict[current_col_idx] = cell_value
                current_col_idx += 1

            # Convert sparse dict to list, padding with empty strings
            max_col_in_row = max(row_data_dict.keys()) if row_data_dict else -1
            row_list = [row_data_dict.get(i, '') for i in range(max_col_in_row + 1)]
            data_rows.append(row_list)
            max_cols_seen = max(max_cols_seen, len(row_list))

        # Pad data rows to match the maximum number of columns found (header or data)
        # and ensure header length matches final data width
        final_header = list(header) # Make a mutable copy
        if max_cols_seen > len(final_header):
            final_header.extend([f'Column_{i}' for i in range(len(final_header), max_cols_seen)])
        elif max_cols_seen < len(final_header): # Trim header if data is consistently narrower
             final_header = final_header[:max_cols_seen]

        sheet_data_padded = [row + [''] * (max_cols_seen - len(row)) for row in data_rows]

        if sheet_data_padded:
            df_sheet = pd.DataFrame(sheet_data_padded, columns=final_header)
            dataframes.append(df_sheet)

    if not dataframes:
        raise ValueError("Aucune donnée extraite du fichier XML.")

    # Combine all worksheets into one DataFrame
    return pd.concat(dataframes, ignore_index=True)

def upload_file():
    """
    Upload d'un fichier (CSV ou Excel) depuis la machine locale vers Colab.
    Détecte automatiquement le type et le lit.
    Gère spécifiquement les fichiers XML Spreadsheet 2003.
    Retourne le nom du fichier et un DataFrame pandas.
    """
    print("Veuillez sélectionner votre fichier (CSV ou Excel) :")
    uploaded = files.upload()

    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename]
    file_content_bytes = io.BytesIO(file_content)

    # Détection de l'extension
    ext = os.path.splitext(filename)[1].lower()

    df = None
    try:
        if ext == '.csv':
            # Fichier CSV
            df = pd.read_csv(file_content_bytes, encoding='utf-8')
        elif ext in ['.xls', '.xlsx']:
            # Fichier Excel : utiliser openpyxl pour les .xlsx, xlrd pour les .xls
            try:
                # D'abord avec openpyxl (pour xlsx)
                df = pd.read_excel(file_content_bytes, engine='openpyxl')
            except Exception as e_openpyxl:
                print(f"Tentative avec openpyxl échouée, essai avec xlrd. Erreur: {e_openpyxl}")
                # Reset stream position before trying another engine
                file_content_bytes.seek(0)
                try:
                    # Si ça échoue, essayer avec xlrd (pour vieux xls)
                    df = pd.read_excel(file_content_bytes, engine='xlrd')
                except XLRDError as e_xlrd:
                    # Check for the specific XML Spreadsheet 2003 error
                    if "Expected BOF record" in str(e_xlrd) and "<?xml" in str(e_xlrd):
                        print(f"Détecté un fichier XML Spreadsheet 2003 masqué en .xls. Tentative de parsing XML...")
                        file_content_bytes.seek(0)
                        raw_xml_content = file_content_bytes.read().decode('utf-8')
                        df = _parse_xml_spreadsheet_to_dataframe(raw_xml_content)
                    else:
                        raise e_xlrd # Re-raise if it's another type of XLRDError
                except Exception as e_general_excel:
                    raise ValueError(f"Erreur lors de la lecture du fichier Excel avec xlrd : {e_general_excel}")
        else:
            raise ValueError(f"Extension non supportée : {ext}. Utilisez .csv, .xls ou .xlsx")

        # After parsing XML, clean column names to match expected aggregation names
        if df is not None and ext in ['.xls', '.xlsx'] and filename.lower().endswith('.xls') and '<?xml' in str(file_content[:50]): # Rough check for XML content
            df.columns = df.columns.str.strip().str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
            # Specific renames for common XML Spreadsheet column names
            rename_map = {
                'Temp_rature____C_': 'temperature',
                'Point_de_ros_e___C_': 'point rosee',
                'Rayonnement_solaire__W_m2_': 'rayonnement',
                'D_ficit_de_Pression_de_vapeur__kPa_': 'deficit de pression de vapeur',
                'Humidit__Relative_____' : 'humidite relative',
                'Pr_cipitations__mm_': 'precipitation',
                'Vitesse_du_vent__m_s_': 'vitesse du vent',
                'Direction_du_vent__deg_': 'direction du vent',
                'Panneau_solaire__mV_': 'pannau solaire batterie' # Assuming this is the closest match
            }
            df.rename(columns=rename_map, inplace=True)
            # Handle 'Date/heure' or 'Column_0' if not already renamed
            if 'Column_0' in df.columns and 'Date/heure' not in df.columns:
                df.rename(columns={'Column_0': 'Date/heure'}, inplace=True)
            # Handle 'et0' if it comes as a generic Column_X
            # This is a bit of a guess, based on common placements
            if 'Column_24' in df.columns and 'et0' not in df.columns:
                df.rename(columns={'Column_24': 'et0'}, inplace=True)

        if df is None: # Should not happen if all paths handled
            raise ValueError("Le fichier n'a pas pu être lu par aucun des moteurs supportés.")

        print(f"✅ Fichier '{filename}' chargé avec succès. Dimensions : {df.shape}")
        return filename, df
    except Exception as e:
        print(f"❌ Erreur de lecture du fichier : {e}")
        raise

def aggregate_to_monthly(df, date_col='Date/heure'):
    """
    Agrège les données horaires en données mensuelles.
    """
    df = df.copy()

    # Convertir la colonne date en datetime (gérer plusieurs formats)
    try:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    except:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)

    # Supprimer les lignes invalides
    initial_rows = len(df)
    df = df.dropna(subset=[date_col])
    if len(df) < initial_rows:
        print(f"⚠️ {initial_rows - len(df)} lignes supprimées car date invalide.")

    # Créer année et mois
    df['année'] = df[date_col].dt.year
    df['mois'] = df[date_col].dt.month

    # Configuration des agrégations
    agg_config = {
        'temperature': 'mean',
        'point rosee': 'mean',
        'rayonnement': 'mean',
        'deficit de pression de vapeur': 'mean',
        'humidite relative': 'mean',
        'precipitation': 'sum',
        'vitesse du vent': 'mean',
        'direction du vent': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
        'pannau solaire batterie': 'mean',
        'et0': 'sum'
    }

    # Mapping des colonnes (ignorer casse et espaces)
    col_mapping = {}
    for standard_name in agg_config.keys():
        for col in df.columns:
            if col.lower().strip() == standard_name.lower().strip():
                col_mapping[col] = standard_name
                break

    found_cols = list(col_mapping.values())
    missing = set(agg_config.keys()) - set(found_cols)
    if missing:
        print(f"⚠️ Colonnes non trouvées : {missing}")

    # Construire le dictionnaire d'agrégation
    agg_dict = {}
    for original_col, standard_name in col_mapping.items():
        agg_dict[original_col] = agg_config[standard_name]

    if not agg_dict:
        raise ValueError("Aucune colonne météo reconnue. Vérifiez les noms.")

    # Agrégation
    grouped = df.groupby(['année', 'mois']).agg(agg_dict).reset_index()

    # Renommer les colonnes
    rename_dict = {orig: std for orig, std in col_mapping.items()}
    grouped = grouped.rename(columns=rename_dict)

    # Créer index date
    grouped['date'] = pd.to_datetime(grouped['année'].astype(str) + '-' + grouped['mois'].astype(str), format='%Y-%m')
    grouped = grouped.set_index('date').drop(columns=['année', 'mois'])

    # Réordonner
    ordered_cols = [c for c in agg_config.keys() if c in grouped.columns]
    grouped = grouped[ordered_cols]

    return grouped

def save_monthly_data(df_monthly, original_filename):
    """
    Sauvegarde et télécharge le fichier mensuel.
    """
    base = os.path.splitext(original_filename)[0]
    output_filename = f"{base}_mensuel.csv"
    df_monthly.to_csv(output_filename, encoding='utf-8-sig')
    print(f"\n✅ Fichier mensuel créé : {output_filename}")
    files.download(output_filename)
    return output_filename

# ==================== EXÉCUTION ====================
if __name__ == "__main__":
    print("=== Agrégation horaire → mensuelle (CSV/Excel) ===\n")

    # 1. Upload
    filename, df_raw = upload_file()

    # 2. Aperçu
    print("\n--- Aperçu des 5 premières lignes ---")
    print(df_raw.head())
    print("\n--- Colonnes disponibles ---")
    print(df_raw.columns.tolist())

    # --- PREPROCESSING STEPS FOR df_raw ---
    # Remove the first row which contains sub-headers ('moy', 'max', 'min')
    df_processed = df_raw.iloc[1:].copy()

    # Convert relevant columns to numeric, coercing errors
    numeric_cols_to_convert = [
        'temperature', 'point rosee', 'rayonnement', 'deficit de pression de vapeur',
        'humidite relative', 'precipitation', 'vitesse du vent', 'pannau solaire batterie', 'et0'
    ]
    for col in numeric_cols_to_convert:
        if col in df_processed.columns:
            df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

    print("\n--- Aperçu des 5 premières lignes après prétraitement ---")
    print(df_processed.head())
    print("\n--- Types de colonnes après prétraitement ---")
    print(df_processed[numeric_cols_to_convert].dtypes)
    # --- END PREPROCESSING ---

    # 3. Agrégation (si colonne date existe)
    if 'Date/heure' not in df_processed.columns:
        print("\n⚠️ La colonne 'Date/heure' n'existe pas. Les colonnes trouvées :")
        print(df_processed.columns.tolist())
        # Option : essayer de détecter une colonne contenant 'date' ou 'heure'
        possible_date = [c for c in df_processed.columns if 'date' in c.lower() or 'heure' in c.lower()]
        if possible_date:
            print(f"Utilisation de '{possible_date[0]}' comme colonne date.")
            date_col = possible_date[0]
        else:
            raise ValueError("Ajoutez une colonne nommée 'Date/heure' ou renommez votre colonne.")
    else:
        date_col = 'Date/heure'

    try:
        df_monthly = aggregate_to_monthly(df_processed, date_col=date_col)
        print("\n--- Données mensuelles obtenues ---")
        print(df_monthly.round(2))

        # 4. Sauvegarde et téléchargement
        save_monthly_data(df_monthly, filename)

        print("\n--- Résumé statistique mensuel ---")
        print(df_monthly.describe())

    except Exception as e:
        print(f"\n❌ Erreur : {e}")

=== Agrégation horaire → mensuelle (CSV/Excel) ===

Veuillez sélectionner votre fichier (CSV ou Excel) :


Saving 2024.xls to 2024 (2).xls
Tentative avec openpyxl échouée, essai avec xlrd. Erreur: File is not a zip file
Détecté un fichier XML Spreadsheet 2003 masqué en .xls. Tentative de parsing XML...
✅ Fichier '2024 (2).xls' chargé avec succès. Dimensions : (8761, 25)

--- Aperçu des 5 premières lignes ---
            Date/heure temperature point rosee rayonnement  \
0           Date/heure         moy         max         min   
1  2024-12-31 15:00:00        None        None        None   
2  2024-12-31 14:00:00        None        None        None   
3  2024-12-31 13:00:00        None        None        None   
4  2024-12-31 12:00:00        None        None        None   

  deficit de pression de vapeur humidite relative precipitation  \
0                           moy               min           moy   
1                          None              None           412   
2                          None              None           486   
3                          None              None     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Résumé statistique mensuel ---
       temperature  point rosee  rayonnement  deficit de pression de vapeur  \
count    12.000000    12.000000    12.000000                      12.000000   
mean     19.722116    20.284912    19.166896                      14.832349   
std       4.921207     4.847044     4.995671                       3.689660   
min      12.511132    13.039245    11.940440                       9.801786   
25%      15.141389    15.766253    14.527093                      12.103484   
50%      20.834408    21.352983    20.328511                      13.556460   
75%      23.440510    23.931635    22.951781                      18.273811   
max      25.733266    26.251653    25.218522                      20.453226   

       humidite relative  precipitation  vitesse du vent  \
count          12.000000      12.000000        12.000000   
mean           14.233215  139691.833333         0.709545   
std             3.793832   46174.870436         0.448617   
min         